# Neural Machine Translation

- Input is a sentence (sequence) in Spanish --> English
- Output is the corresponding sequence in Mexican Sign Language Glosses --> German
- Encoder Decoder model with a Bidirectional GRU Encoder, Attention and GRU Decoder

## Import needed libraries

In [6]:
import tensorflow as tf
import numpy as np

# Import local libraries
import src.text_processing as text_processing
import src.dictionary as dictionary
import src.neural_network as neural_network

# Update python files
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data processing

### Read dataset

In [2]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /home/wangkongqiang/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [7]:
# Read file containing english and german translations
data = text_processing.load_doc("./dataset/SPA_to_MSLG.txt")

# Split data into english and german
english_sentences, german_sentences = text_processing.prepare_data(data)

# Check and print number of sentences from one language to the other
assert(len(english_sentences) == len(german_sentences))
print(english_sentences.shape)

# Example of sentence with translation
print(english_sentences[20])
print(german_sentences[20])

line_split: ['Los peces azules son mis favoritos.', 'PEZ COLOR AZUL MI FAVORITO']
line_split: ['Estoy feliz, mi hijo ya tiene licencia de conducir.', 'YO FELIZ MI HIJO LICENCIA-DE-CONDUCIR ÉL YA TENER']
line_split: ['Saquen su libro de Matemáticas.', 'SU LIBRO MATEMÁTICAS USTEDES SACAR']
line_split: ['Necesito papel para absorber el agua.', 'YO NECESITAR PAPEL PARA AGUA ABSORBER']
line_split: ['Isabel tiene una corona de oro.', 'dm-ISABEL TENER SU CORONA ORO']
line_split: ['Este juguete está barato.', 'ESTE JUGUETE BARATO']
line_split: ['¿Le pusiste atención a tu maestra?', '¿TUYA MAESTRO+MUJER TÚ PONER-ATENCIÓN ELLA?']
line_split: ['Mi mamá usa crema para la cara.', 'MI MAMÁ USA CREMA-DE-COSMÉTICO']
line_split: ['El bebé siempre es muy expresivo.', 'SIEMPRE BEBÉ EXPRESAR EXPRESAR']
line_split: ['Ayer llegó mi tío de San Francisco.', 'AYER MI TIO SAN FRANCISCO YA LLEGAR']
line_split: ['Debes pagarme a fuerza.', 'A-FUERZA TÚ DEBER TÚ PAGAR MÍ']
line_split: ['Esa antena es muy cara.', 'E

/mnt/c/Users/wangkongqiang/PycharmProjects/pythonProject2/IberLEF2026_MSLG-SPA/Neural_Machine_Translation/src/text_processing.py:115: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(english_sentences), np.array(german_sentences)


### Split dataset (training + validation)

In [8]:
# Split percentage of training and validation
split_percentage = 0.85

# Count how many samples into training dataset
total_dataset = len(english_sentences)
train_dataset = int(total_dataset * split_percentage)

# Set random seed to have always same training and validation split
np.random.seed(42)
train_indices = np.random.choice(total_dataset, train_dataset, replace=False)

# Get training data for the two languages
training_english = english_sentences[train_indices]
training_german = german_sentences[train_indices]

# Get validation data
validation_english = np.delete(english_sentences, train_indices)
validation_german = np.delete(german_sentences, train_indices)

print("Training samples: " + str(training_english.shape[0]))
print("Validation samples: " + str(validation_english.shape[0]))

Training samples: 416
Validation samples: 74


### Create dictionaries for the two languages

In [9]:
# Calculate longest sentence in the two languages
english_max_length = text_processing.max_length_sentence(training_english)
german_max_length = text_processing.max_length_sentence(training_german) + 2  # + 2 because of <START> and <END> the beginning

print("Longest sentence in SPA-->English has " + str(english_max_length) + " tokens.")
print("Longest sentence in MSLG-->German has " + str(german_max_length) + " tokens.")
print()

# Create dictionaries
english_dictionary = dictionary.LanguageDictionary(training_english, english_max_length)
german_dictionary = dictionary.LanguageDictionary(training_german, german_max_length)

# Calculate size of the dictionaries
english_dictionary_size = len(english_dictionary.index_to_word)
german_dictionary_size = len(german_dictionary.index_to_word)

print("SPA-->English dictionary size: " + str(english_dictionary_size))
print("MSLG-->German dictionary size: " + str(german_dictionary_size))

# Save dictionaries
text_processing.save_dump(english_dictionary, "./dumps/eng_SPA_2_dict.pickle")
text_processing.save_dump(german_dictionary, "./dumps/ger_MSLG_2_dict.pickle")

Longest sentence in SPA-->English has 14 tokens.
Longest sentence in MSLG-->German has 13 tokens.

SPA-->English dictionary size: 1177
MSLG-->German dictionary size: 958


### Prepare sequences for the Neural Network

In [10]:
# Prepare sequences of training data
train_source_input, train_target_input = text_processing.prepare_sequences(training_english, 
                                                                           training_german, 
                                                                           english_dictionary, 
                                                                           german_dictionary)

# Prepare sequences of validation data
val_source_input, val_target_input = text_processing.prepare_sequences(validation_english, 
                                                                       validation_german, 
                                                                       english_dictionary, 
                                                                       german_dictionary)

# Check if same number of samples
assert(len(train_source_input) == len(train_target_input))
assert(len(val_source_input) == len(val_target_input))

# Print shapes data
print("Training samples : " + str(len(train_source_input)))
print(train_source_input.shape)
print(train_target_input.shape)

print("Validation samples : " + str(len(val_source_input)))
print(val_source_input.shape)
print(val_target_input.shape)

Training samples : 416
(416, 14)
(416, 13)
Validation samples : 74
(74, 14)
(74, 13)


### Print sample input data in SPA-->English, MSLG-->German and next word to be predicted in MSLG-->German

In [11]:
print(train_source_input[0])
print(train_target_input[0])

print("SOURCE => " + english_dictionary.indices_to_text(train_source_input[0]))
print("TARGET => " + german_dictionary.indices_to_text(train_target_input[0]))

[0 0 0 0 0 0 0 0 0 0 4 5 6 7]
[1 4 5 2 0 0 0 0 0 0 0 0 0]
SOURCE => <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> El jabón hace burbujas
TARGET => <START> JABÓN BURBUJAS <END> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>


## Neural Network

### Parameters

In [12]:
epochs = 200
batch_size = 32
embedding_size = 256
lstm_hidden_units = 192
lr = 1e-3
keep_dropout_prob = 0.7

### Create Seq2seq neural network graph

In [13]:
tf.reset_default_graph()

# Placeholders
input_sequence = tf.placeholder(tf.int32, (None, english_dictionary.max_length_sentence), 'inputs')
output_sequence = tf.placeholder(tf.int32, (None, None), 'output')
target_labels = tf.placeholder(tf.int32, (None, None), 'targets')
keep_prob = tf.placeholder(tf.float32, (None), 'dropout_prob')
decoder_outputs_tensor = tf.placeholder(tf.float32, (None, german_dictionary.max_length_sentence - 1, 
                                                     lstm_hidden_units * 2), 'output')

# Create graph for the network
logits, dec_output, mask = neural_network.create_network(input_sequence, 
                                                         output_sequence, 
                                                         keep_prob,
                                                         decoder_outputs_tensor,
                                                         english_dictionary_size, 
                                                         german_dictionary_size, 
                                                         embedding_size,
                                                         lstm_hidden_units)

Previous decoder outputs:  Tensor("decoder/ExpandDims:0", shape=(?, 12, 1, 384), dtype=float32)
Bahdanau score:  Tensor("decoder/dense_2/BiasAdd:0", shape=(?, 12, 14, 1), dtype=float32)
Attention weights:  Tensor("decoder/transpose_1:0", shape=(?, 12, 14, 1), dtype=float32)
Context vector:  Tensor("decoder/Sum:0", shape=(?, 12, 384), dtype=float32)
Embedding layer:  Tensor("decoder/embedding_lookup/Identity:0", shape=(?, ?, 256), dtype=float32)
Decoder input:  Tensor("decoder/concat_2:0", shape=(?, 12, 640), dtype=float32)
Logits: Tensor("dense/BiasAdd:0", shape=(?, 12, 958), dtype=float32)


### Set the loss function, optimizer and other useful tensors

In [14]:
# Cross entropy loss after softmax of logits
ce = tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=target_labels) * mask
loss = tf.reduce_mean(ce)

# Using Adam optimizer for the update of the weights of the network with gradient clipping
optimizer = tf.train.AdamOptimizer(learning_rate=lr) #.minimize(loss)
gradients, variables = zip(*optimizer.compute_gradients(loss))
gradients, _ = tf.clip_by_global_norm(gradients, 5.0)
optimize = optimizer.apply_gradients(zip(gradients, variables))

# Useful tensors
scores = tf.nn.softmax(logits)
predictions = tf.to_int32(tf.argmax(scores, axis=2))
correct_mask = tf.to_float(tf.equal(predictions, target_labels))
accuracy = tf.contrib.metrics.accuracy(predictions, target_labels, weights=mask)


Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Deprecated in favor of operator or tf.math.divide.


### Training of the network

In [15]:
# Training and validation data variables
training_overfit = False
best_val_accuracy = 0
consecutive_validation_without_saving = 0
indices = list(range(len(train_source_input)))
print("Number of iterations per epoch: " + str((len(train_source_input) // batch_size) + 1))

# Start session and initialize variables in the graph
with tf.Session() as sess:    
    
    saver = tf.train.Saver()
    sess.run(tf.global_variables_initializer())
    
    for i in range(epochs):
        
        # Vector accumulating accuracy and loss during one epoch
        total_accuracies, total_losses = [], []
        
        # Shuffle data to not train the network always with the same order
        np.random.shuffle(indices)
        train_source_input = train_source_input[indices]
        train_target_input = train_target_input[indices]        
        
        # Iterate over mini-batches
        for j in range(0, len(train_source_input), batch_size):

            dec_out_tmp = neural_network.get_decoder_outputs(sess, dec_output, input_sequence, output_sequence,
                        decoder_outputs_tensor, keep_prob, keep_dropout_prob, 
                        len(train_source_input[j:j+batch_size]), german_dictionary.max_length_sentence - 1, 
                        lstm_hidden_units, train_source_input[j:j+batch_size],
                        train_target_input[j:j+batch_size, :-1])
            
            _, avg_accuracy, avg_loss = sess.run([optimize, accuracy, loss], feed_dict={
                                                input_sequence: train_source_input[j:j+batch_size],
                                                output_sequence: train_target_input[j:j+batch_size, :-1],
                                                target_labels: train_target_input[j:j+batch_size, 1:],
                                                keep_prob: keep_dropout_prob,
                                                decoder_outputs_tensor: dec_out_tmp })
            
            # Add values for this mini-batch iterations
            total_losses.append(avg_loss) 
            total_accuracies.append(avg_accuracy)
            
            # Statistics on validation set
            if (j // batch_size + 1) % 5 == 0:

                # Accumulate validation statistics
                val_accuracies, val_losses = [], []
                for k in range(0, len(val_source_input), batch_size):

                    dec_out_tmp = neural_network.get_decoder_outputs(sess, dec_output, input_sequence,
                        output_sequence, decoder_outputs_tensor, keep_prob, 1.0,
                        len(val_source_input[k:k+batch_size]), german_dictionary.max_length_sentence - 1, 
                        lstm_hidden_units, val_source_input[k:k+batch_size], val_target_input[k:k+batch_size, :-1])
                    
                    avg_accuracy, avg_loss = sess.run([accuracy, loss], feed_dict={
                                            input_sequence: val_source_input[k:k+batch_size],
                                            output_sequence: val_target_input[k:k+batch_size, :-1],
                                            target_labels: val_target_input[k:k+batch_size, 1:],
                                            keep_prob: 1.0,
                                            decoder_outputs_tensor: dec_out_tmp })                    
                    
                    val_losses.append(avg_loss) 
                    val_accuracies.append(avg_accuracy)
            
                # Average validation accuracy over batches
                final_val_accuracy = np.mean(val_accuracies)
                
                # Save model if validation accuracy better
                if final_val_accuracy > best_val_accuracy:
                    consecutive_validation_without_saving = 0
                    best_val_accuracy = final_val_accuracy
                    print("VALIDATION loss: " + str(np.mean(val_losses)) + ", accuracy: " + str(final_val_accuracy))
                    save_path = saver.save(sess, "./SPA_MSLG_checkpoints/model.ckpt")
                else:
                    # Count every time check validation accuracy
                    consecutive_validation_without_saving += 1
                
                # If checked validation time many consecutive times without having improvement in accuracy
                if consecutive_validation_without_saving >= 10:
                    training_overfit = True
                    break
        
        # Epoch statistics
        print("Epoch: " + str(i+1) + ", AVG loss: " + str(np.mean(np.array(total_losses))) + 
              ", AVG accuracy: " + str(np.mean(np.array(total_accuracies))) + "\n")
        
        if training_overfit:
            print("Early stopping")
            break

Number of iterations per epoch: 14


2026-04-17 15:55:34.047744: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1159] Device interconnect StreamExecutor with strength 1 edge matrix:
2026-04-17 15:55:34.047778: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1165]      


VALIDATION loss: 2.8871727, accuracy: 0.25045517
VALIDATION loss: 2.6957676, accuracy: 0.30107495
Epoch: 1, AVG loss: 3.086842, AVG accuracy: 0.2385525

VALIDATION loss: 2.7402556, accuracy: 0.32403362
VALIDATION loss: 2.763192, accuracy: 0.32849792
Epoch: 2, AVG loss: 2.624495, AVG accuracy: 0.31447613

Epoch: 3, AVG loss: 2.4828517, AVG accuracy: 0.3266396

VALIDATION loss: 2.7697585, accuracy: 0.33970937
Epoch: 4, AVG loss: 2.3802633, AVG accuracy: 0.33671123

VALIDATION loss: 2.7472963, accuracy: 0.34417367
Epoch: 5, AVG loss: 2.2696216, AVG accuracy: 0.34461686

VALIDATION loss: 2.7621405, accuracy: 0.3522304
VALIDATION loss: 2.7620573, accuracy: 0.3535399
Epoch: 6, AVG loss: 2.1448646, AVG accuracy: 0.35481054

Epoch: 7, AVG loss: 2.0140133, AVG accuracy: 0.3679687

Epoch: 8, AVG loss: 1.8608739, AVG accuracy: 0.39050007

VALIDATION loss: 2.686886, accuracy: 0.35389706
Epoch: 9, AVG loss: 1.7078124, AVG accuracy: 0.41286528

VALIDATION loss: 2.744105, accuracy: 0.355028
Epoch: 10

## Testing network

### Rebuild graph quickly if want to run only this part of the notebook

In [16]:
# Load dictionaries from pickle
english_dictionary = text_processing.load_dump("./dumps/eng_SPA_2_dict.pickle")
german_dictionary = text_processing.load_dump("./dumps/ger_MSLG_2_dict.pickle")

tf.reset_default_graph()

embedding_size = 256
lstm_hidden_units = 192

# Placeholders
input_sequence = tf.placeholder(tf.int32, (None, english_dictionary.max_length_sentence), 'inputs')
output_sequence = tf.placeholder(tf.int32, (None, None), 'output')
target_labels = tf.placeholder(tf.int32, (None, None), 'targets')
keep_prob = tf.placeholder(tf.float32, (None), 'dropout_prob')
decoder_outputs_tensor = tf.placeholder(tf.float32, (None, german_dictionary.max_length_sentence - 1, 
                                                     lstm_hidden_units * 2), 'output')

# Create graph for the network
logits, dec_output, mask = neural_network.create_network(input_sequence, 
                                                         output_sequence, 
                                                         keep_prob,
                                                         decoder_outputs_tensor,
                                                         len(english_dictionary.index_to_word), 
                                                         len(german_dictionary.index_to_word), 
                                                         embedding_size,
                                                         lstm_hidden_units)
# Predictions
scores = tf.nn.softmax(logits)
predictions = tf.to_int32(tf.argmax(scores, axis=2))

Previous decoder outputs:  Tensor("decoder/ExpandDims:0", shape=(?, 12, 1, 384), dtype=float32)
Bahdanau score:  Tensor("decoder/dense_2/BiasAdd:0", shape=(?, 12, 14, 1), dtype=float32)
Attention weights:  Tensor("decoder/transpose_1:0", shape=(?, 12, 14, 1), dtype=float32)
Context vector:  Tensor("decoder/Sum:0", shape=(?, 12, 384), dtype=float32)
Embedding layer:  Tensor("decoder/embedding_lookup/Identity:0", shape=(?, ?, 256), dtype=float32)
Decoder input:  Tensor("decoder/concat_2:0", shape=(?, 12, 640), dtype=float32)
Logits: Tensor("dense/BiasAdd:0", shape=(?, 12, 958), dtype=float32)


### Perform test predictions

In [26]:
with tf.Session() as sess:
    saver = tf.train.Saver()
    sess.run(tf.global_variables_initializer())
    saver.restore(sess, "./SPA_MSLG_checkpoints/model.ckpt") 

    test_source_sentence = ["Los peces azules son mis favoritos.","Estoy feliz, mi hijo ya tiene licencia de conducir."]

    for source_sentence in test_source_sentence:
        
        # Normalize & tokenize (cut if longer than max_length_source)  
        source_preprocessed = text_processing.preprocess_sentence(source_sentence)
        
        # Convert to numbers
        source_encoded = english_dictionary.text_to_indices(source_preprocessed)
        
        # Add padding
        source_input = text_processing.pad_sentence(source_encoded, english_dictionary.max_length_sentence)
        
        # Starting target sentence in German
        target_sentence = [["<START>"]]
        target_encoded = german_dictionary.text_to_indices(target_sentence[0])

        i = 0
        word_predicted = 0
        while word_predicted != 2: # If <END> (index 2), stop
            
            target_encoded_pad = text_processing.pad_sentence(target_encoded, 
                                                          german_dictionary.max_length_sentence - 1, 
                                                           pad_before=False)
            
            dec_out_tmp = neural_network.get_decoder_outputs(
                                                            sess,
                                                            dec_output,
                                                            input_sequence,
                                                            output_sequence,
                                                            decoder_outputs_tensor,
                                                            keep_prob,
                                                            1.0,
                                                            1, 
                                                            german_dictionary.max_length_sentence - 1, 
                                                            lstm_hidden_units,
                                                            [source_input],
                                                            [target_encoded_pad])        
            # Perform prediction
            pred = sess.run(predictions, feed_dict={ input_sequence: [source_input], 
                                                    output_sequence: [target_encoded_pad],
                                                    keep_prob: 1.0,
                                                    decoder_outputs_tensor: dec_out_tmp })
            
            # Accumulate
            target_encoded.append(pred[0][i])
            word_predicted = pred[0][i]
            
            if i > german_dictionary.max_length_sentence:
                break
            i += 1

        print(english_dictionary.indices_to_text(source_input) + " => "
              + german_dictionary.indices_to_text(target_encoded))

INFO:tensorflow:Restoring parameters from ./SPA_MSLG_checkpoints/model.ckpt


2026-04-17 16:34:09.818691: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1159] Device interconnect StreamExecutor with strength 1 edge matrix:
2026-04-17 16:34:09.818721: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1165]      


<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> Los peces azules son mis favoritos => <START> PEZ COLOR AZUL MI FAVORITO <END>
<PAD> <PAD> <PAD> <PAD> <PAD> Estoy feliz mi hijo ya tiene licencia de conducir => <START> YO FELIZ MI HIJO LICENCIA-DE-CONDUCIR ÉL YA TENER <END>


很好，这个需求很明确： 👉 把原来 print 的结果，改成写入 .txt 文件，并严格符合提交格式

例如：

In [27]:
def load_test_file(file_path):
    """
    Load test file with format:
    ID    MSLG
    287   ELLA TENER BUSTO PEQUEÑO

    Returns:
        sentences: List[str]
        ids: List[str]
    """
    sentences = []
    ids = []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        # ❗跳过表头
        if i == 0 and ("ID" in line and "SPA" in line):
            continue

        # 👉 用 split（支持多个空格或tab）
        parts = line.split()

        if len(parts) < 2:
            continue

        instance_id = parts[0]
        print("instance_id:", instance_id)
        sentence = " ".join(parts[1:])
        print("sentence:", sentence)
        ids.append(instance_id)
        sentences.append(sentence)

    return sentences, ids

In [29]:
with tf.Session() as sess:
    saver = tf.train.Saver()
    sess.run(tf.global_variables_initializer())
    saver.restore(sess, "./SPA_MSLG_checkpoints/model.ckpt") 

    # test_source_sentence = ["PEZ COLOR AZUL MI FAVORITO"]
    test_source_sentence, instance_ids = load_test_file("SPA2MSLG_test.txt")

    # 👉 输出文件
    # output_file = "TeamName_SolutionName_MSLG2SPA.txt"
    output_file = "result/wangkongqiang_NeuralMachineTranslation_SPA2MSLG.txt"
    with open(output_file, "w", encoding="utf-8") as f:

        for idx, source_sentence in enumerate(test_source_sentence):
            instance_id = instance_ids[idx]
            # Normalize & tokenize
            source_preprocessed = text_processing.preprocess_sentence(source_sentence)
            
            # Convert to numbers
            source_encoded = english_dictionary.text_to_indices(source_preprocessed)
            
            # Add padding
            source_input = text_processing.pad_sentence(
                source_encoded, english_dictionary.max_length_sentence
            )
            
            # Starting target sentence
            target_sentence = [["<START>"]]
            target_encoded = german_dictionary.text_to_indices(target_sentence[0])

            i = 0
            word_predicted = 0

            while word_predicted != 2:  # <END>
                
                target_encoded_pad = text_processing.pad_sentence(
                    target_encoded,
                    german_dictionary.max_length_sentence - 1,
                    pad_before=False
                )
                
                dec_out_tmp = neural_network.get_decoder_outputs(
                    sess,
                    dec_output,
                    input_sequence,
                    output_sequence,
                    decoder_outputs_tensor,
                    keep_prob,
                    1.0,
                    1,
                    german_dictionary.max_length_sentence - 1,
                    lstm_hidden_units,
                    [source_input],
                    [target_encoded_pad]
                )        

                pred = sess.run(
                    predictions,
                    feed_dict={
                        input_sequence: [source_input],
                        output_sequence: [target_encoded_pad],
                        keep_prob: 1.0,
                        decoder_outputs_tensor: dec_out_tmp
                    }
                )
                
                # target_encoded.append(pred[0][i])
                # word_predicted = pred[0][i]
                
                 # ✅ 关键修复：取当前最后一个位置
                next_index = len(target_encoded) - 1

                if next_index >= len(pred[0]):
                    break

                word_predicted = pred[0][next_index]
                target_encoded.append(word_predicted)

                # ✅ 防止死循环
                if len(target_encoded) >= german_dictionary.max_length_sentence:
                    break
                    
                # if i > german_dictionary.max_length_sentence:
                #     break
                i += 1

            # 👉 解码结果
            output_text = german_dictionary.indices_to_text(target_encoded)

            # 👉 去掉 <START> <END>（很重要）
            output_text = output_text.replace("<START>", "").replace("<END>", "").strip()

            # 👉 写入文件（严格格式）
            # instance_id = str(idx + 1)
            line = f"\"{instance_id}\"\t\"{output_text}\"\n"
            f.write(line)

    print(f"Predictions saved to {output_file}")

INFO:tensorflow:Restoring parameters from ./SPA_MSLG_checkpoints/model.ckpt


2026-04-17 16:40:46.357838: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1159] Device interconnect StreamExecutor with strength 1 edge matrix:
2026-04-17 16:40:46.357878: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1165]      


instance_id: 476
sentence: Esa marca es excelente.
instance_id: 707
sentence: El gobernador apoyó la construcción de una escuela para alumnos sordos.
instance_id: 443
sentence: Javier es mi concuño.
instance_id: 665
sentence: Yo compré una camisa elegante.
instance_id: 661
sentence: El bikini es azul.
instance_id: 115
sentence: Debemos comparar antes de elegir.
instance_id: 59
sentence: Ángela le gusta el ballet desde niña
instance_id: 582
sentence: El joven es hábil en el kayak.
instance_id: 447
sentence: Mi consuegro horneó un pastel.
instance_id: 960
sentence: Las personas sordas alaban a San Judas.
instance_id: 30
sentence: Ana es muy amable.
instance_id: 794
sentence: Necesito un taladro para poder arreglarlo.
instance_id: 126
sentence: Es muy difícil convencer a mi maestra.
instance_id: 654
sentence: En Tepito hay muchos asaltos.
instance_id: 25
sentence: ¡Aléjate! Estoy enojada.
instance_id: 327
sentence: Mi cuñada tiene su cabello chino.
instance_id: 169
sentence: Me pegué en l